# persona-v1 — tam validation seti, iki checkpoint

`persona-qlora` eğitimi temiz bitti (48/48 adım, deadline'a takılmadan) ve
30 satırlık ölçümde şunu verdi:

| metrik | taban | adapter |
|---|---:|---:|
| `citation_valid` | 1.00 | 1.00 |
| `grounded_format` | 15/23 | **22/23** |
| `decision_match` | 4/23 | **15/23** |
| `asked_when_thin` | **3/7** | **0/7** |

Üç metrik yükseldi, biri sıfırlandı — ve sıfırlanan, ürünün var olma sebebi olan
davranış: kanıt inceyken tahmin yerine soru sormak. Üç kazancın hepsi tek bir
davranıştan geliyor gibi görünüyor: **her zaman karar ver.**

Bu notebook eğitmiyor. İki soruyu cevaplıyor, ve ikisi de eğitimden ucuz.

## 1. O sıfır gerçek mi

`asked_when_thin` 30 satırın yalnız **7'sinde** ölçüldü, çünkü clarify satırları
sette azınlık. 0/7 yönü tartışmasız ama bir oran değil. Burada **tüm 100
satırlık validation seti** koşuluyor, yani ~28 clarify satırı — payda dört katına
çıkıyor ve sayı bir orana dönüşüyor.

## 2. Çökme ne zaman oldu

`--save-steps 10` iki checkpoint bıraktı: **40** ve **48**. 48 son hâl; 40, sekiz
adım öncesi. Azınlık davranışının çökmesi kademeliyse 40'ta hâlâ soru soruyor
olabilir, ve o zaman elimizde yayına alınabilir bir build var demektir — yeni bir
eğitim koşusu değil, daha erken bir durak.

İkisi de ölçülüyor çünkü ikisi de zaten diskte. Ölçmemenin maliyeti, cevabı
bilmek için üç saatlik bir koşuyu tekrar etmek olurdu.

## Bütçe

`persona-qlora`'nın ölçülen hızından: 30 satır + model yüklemesi = 6,5 dk.

| aşama | tahmin |
|---|---|
| kurulum + model indirme | ~11 dk |
| taban, 100 satır | ~15 dk |
| checkpoint-40, 100 satır | ~15 dk |
| checkpoint-48, 100 satır | ~15 dk |
| **toplam** | **~56 dk** |

Eğitim yok, yani deadline'a da gerek yok.

In [ ]:
import glob, json, os, shutil, subprocess, sys
import torch

assert torch.cuda.is_available(), "GPU acik degil - Settings > Accelerator > GPU T4"
cap = torch.cuda.get_device_capability(0)
print("GPU:", torch.cuda.get_device_name(0), "sm_%d%d" % cap)
assert cap >= (7, 5), f"sm_{cap[0]}{cap[1]} yetersiz - T4 (sm_75) gerekiyor"

In [ ]:
!pip -q install -U "transformers>=4.51" "peft>=0.11" "bitsandbytes>=0.43" "accelerate>=0.30" datasets 2>&1 | tail -2
import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__)
# torchao kaldiriliyor, yukseltilmiyor. peft'in LoRA dispatcher'i sardigi her
# kuantize OLMAYAN Linear icin is_torchao_available() soruyor ve uyumsuz surumde
# False donmek yerine ImportError firlatiyor. Bu notebook fp16 yukluyor, yani
# tam da dispatcher'a varilan kol — rubric-curve-eval burada durmustu, taban
# olcumu bittikten SONRA, adapter gecisinin ilk saniyesinde.
!pip -q uninstall -y torchao 2>&1 | tail -1

In [ ]:
def find_mount(slug, marker):
    '''Locate one input mount by the dataset/kernel slug in its path.

    Not by filename. persona-qlora is attached with kernel_sources, which
    contributes the whole of its /kaggle/working — including its own copies of
    the data files. Searching for a data file therefore finds two mounts and
    picks between them by luck; the slug is the only thing that tells them apart.
    '''
    hits = [p for p in glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
            if slug.split("/")[-1] in p]
    assert hits, (f"'{slug}' bagli degil (aranan: {marker}). "
                  f"Kaggle > Notebook > Add Input.")
    return os.path.dirname(sorted(hits, key=len)[0])


for root, dirs, files in os.walk("/kaggle/input"):
    print(root, "->", sorted(files)[:5], "..." if len(files) > 5 else "")
    if root.count("/") > 7:
        dirs.clear()

In [ ]:
WORK = "/kaggle/working"
DATA = find_mount("emrahik/persona-dataset", "persona_eval.jsonl")
print("veri seti:", DATA)

os.makedirs(f"{WORK}/data", exist_ok=True)
for f in os.listdir(DATA):
    dst = f"{WORK}/data/{f}" if f.endswith(".jsonl") else f"{WORK}/{f}"
    shutil.copy(f"{DATA}/{f}", dst)
os.chdir(WORK)

# The two checkpoints come from the training run's output, not the dataset.
RUN = find_mount("emrahik/persona-qlora", "adapter_model.safetensors")
print("egitim kosusu:", RUN)
CKPTS = {}
for step in (40, 48):
    cand = glob.glob(f"/kaggle/input/**/persona-v1/checkpoint-{step}/adapter_model.safetensors",
                     recursive=True)
    assert cand, f"checkpoint-{step} bulunamadi"
    CKPTS[step] = os.path.dirname(cand[0])
    print(f"  checkpoint-{step}: {CKPTS[step]}")

# The step each checkpoint actually reached is read, not assumed: a directory
# name is a label, trainer_state.json is the record.
for step, path in CKPTS.items():
    st = json.load(open(f"{path}/trainer_state.json"))
    print(f"  checkpoint-{step}: global_step={st['global_step']} "
          f"epoch={st['epoch']:.3f} -> {round(st['epoch'] * 800)} satir gecisi")

needs = subprocess.run([sys.executable, "persona_eval.py", "--help"],
                       capture_output=True, text=True).stdout
for flag in ("--local", "--base-only", "--baseline", "--adapter"):
    assert flag in needs, f"persona_eval.py '{flag}' bilmiyor — dataset eski"
print("\nscript guncel")

## Taban — 100 satır, bir kez

`--base-only` ile ölçülüp kaydediliyor; iki checkpoint de onu `--baseline` ile
okuyacak. Tabanı üç kez koşturmak iki model yüklemesi artı iki tam üretim geçişi
demekti, ve birbirinden farklı çıkabilen üç sayı üretirdi.

In [ ]:
LIMIT = 100          # tum validation seti
BASE = "Qwen/Qwen3-4B-Instruct-2507"

n_clarify = sum(1 for l in open("data/persona_eval_meta.jsonl")
                if json.loads(l)["mode"] == "clarify")
print(f"validation: {LIMIT} satir, {n_clarify} clarify "
      f"-> asked_when_thin bu kadar satir uzerinden olculecek "
      f"(onceki kosuda 7 idi)")

r = subprocess.run([sys.executable, "persona_eval.py",
                    "--local", "--base-only",
                    "--local-base-model", BASE,
                    "--eval", "data/persona_eval.jsonl",
                    "--meta", "data/persona_eval_meta.jsonl",
                    "--limit", str(LIMIT),
                    "--out", "out/base_100.json"])
assert r.returncode == 0, f"taban olcumu coktu (exit {r.returncode})"
print(json.dumps(json.load(open("out/base_100.json"))["before"], indent=2))

## İki checkpoint, aynı tabana karşı

Sıra 40 → 48. Eğer `asked_when_thin` 40'ta yüksek ve 48'de sıfırsa, çökme son
sekiz adımda olmuş demektir ve 40 yayına alınabilir bir aday. İkisi de sıfırsa
sorun adım sayısı değil, eğitim karışımı — clarify satırlarının payı ya da
ağırlığı.

In [ ]:
results = {}
for step in (40, 48):
    print("\n" + "=" * 64)
    print(f"  checkpoint-{step}")
    print("=" * 64)
    out = f"out/ckpt_{step}_100.json"
    r = subprocess.run([sys.executable, "persona_eval.py",
                        "--local",
                        "--local-base-model", BASE,
                        "--adapter", CKPTS[step],
                        "--baseline", "out/base_100.json",
                        "--eval", "data/persona_eval.jsonl",
                        "--meta", "data/persona_eval_meta.jsonl",
                        "--limit", str(LIMIT),
                        "--out", out])
    assert r.returncode == 0, f"checkpoint-{step} olcumu coktu (exit {r.returncode})"
    results[step] = json.load(open(out))

In [ ]:
base = json.load(open("out/base_100.json"))["before"]
keys = ("citation_valid", "grounded_format", "asked_when_thin", "decision_match")

print(f"\n{LIMIT} satir, {n_clarify} clarify\n")
print(f"{'metrik':<20}{'taban':>9}{'ckpt-40':>10}{'ckpt-48':>10}")
print("-" * 49)
for k in keys:
    row = f"{k:<20}{base[k]:>9.2f}" if base[k] is not None else f"{k:<20}{'n/a':>9}"
    for step in (40, 48):
        v = results[step]["after"].get(k)
        row += f"{'n/a' if v is None else f'{v:.2f}':>10}"
    print(row)

print("\nKarar kurali: asked_when_thin tabanin ALTINA dusen bir build yayina")
print("alinmaz — diger uc metrikteki kazanc onun yerine gecmez, cunku ucu de")
print("ayni davranistan geliyor: her zaman karar ver.")

ok = [s for s in (40, 48)
      if results[s]["after"].get("asked_when_thin") is not None
      and base["asked_when_thin"] is not None
      and results[s]["after"]["asked_when_thin"] >= base["asked_when_thin"]]
print(f"\nTabani koruyan checkpoint(ler): {ok if ok else 'YOK'}")
if not ok:
    print("Ikisi de dustuyse sorun adim sayisi degil egitim karisimi:")
    print("clarify satirlari setin %32'si ama decide ciktisi kati bir sablon,")
    print("ve kucuk bir butcede model baskin sekle kilitleniyor.")